# LangGraph 意圖判斷與動態路由實作範例

本 Notebook 示範如何使用 **LangGraph** 建立一個具備「意圖辨識」與「動態路由」的 Agent 工作流。

### 工作流逻辑：
1. **意圖判斷 (Intent Classification)**：分析用戶輸入，判斷是否在服務範圍內 (`in_scope` 或 `out_of_scope`)。
2. **條件式路由 (Conditional Routing)**：
   - 若為 `in_scope`：走向 **資料檢索節點 (Retrieve)** 查詢資料，再由 **生成回覆節點 (Generate)** 回答。
   - 若為 `out_of_scope`：直接走向 **拒絕節點 (Refuse)** 回覆並結束。

## 1. 安裝與載入套件

請確保環境中已安裝 `langgraph` 與 `typing-extensions`（若使用較舊版本 Python）。

In [ ]:
# 如果尚未安裝，請取消註解並執行此行
# !pip install langgraph pydantic

In [ ]:
from typing import Literal, TypedDict, Dict, Any
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

## 2. 定義型別與狀態 (State)

使用 `TypedDict` 定義工作流在節點間傳遞的狀態，確保欄位與型別皆有明確的定義。

In [ ]:
class AgentState(TypedDict):
    user_input: str
    intent: Literal["in_scope", "out_of_scope"]
    retrieved_data: str
    response: str

# 定義結構化輸出的 Pydantic 模型（供 LLM 意圖分類使用）
class IntentAnalysis(BaseModel):
    intent: Literal["in_scope", "out_of_scope"] = Field(
        description="判斷用戶的輸入是否屬於『專案進度查詢』的服務範圍內"
    )

## 3. 定義節點 (Nodes)

這裡模擬 LLM 的結構化輸出與 RAG 檢索邏輯。您可以輕鬆將此處的模擬程式碼替換為實際的 LangChain LLM 呼叫（如 `llm.with_structured_output(IntentAnalysis)`）。

In [ ]:
def classify_intent(state: AgentState) -> Dict[str, Any]:
    """
    判斷用戶意圖的節點
    這裡以關鍵字簡易模擬 LLM 的判斷結果。
    """
    user_query = state["user_input"].lower()
    
    # 模擬 LLM 意圖分析：僅接受與「專案」或「進度」相關的詢問
    if "專案" in user_query or "進度" in user_query:
        detected_intent: Literal["in_scope", "out_of_scope"] = "in_scope"
    else:
        detected_intent = "out_of_scope"
        
    print(f"[Node: classify_intent] 偵測到意圖: {detected_intent}")
    return {"intent": detected_intent}

def retrieve_data(state: AgentState) -> Dict[str, Any]:
    """
    資料檢索節點 (僅在 in_scope 時觸發)
    """
    print("[Node: retrieve_data] 正在從資料庫檢索專案相關資訊...")
    # 模擬 RAG 或資料庫檢索結果
    mock_db_result = "專案 Alpha 目前進度為 85%，預計下週進行 Beta 測試。"
    return {"retrieved_data": mock_db_result}

def generate_reply(state: AgentState) -> Dict[str, Any]:
    """
    整合檢索資料並產生回覆的節點
    """
    print("[Node: generate_reply] 正在生成最終答覆...")
    final_response = f"根據為您查詢到的系統資料：\n『{state['retrieved_data']}』\n請您參考。"
    return {"response": final_response}

def refuse(state: AgentState) -> Dict[str, Any]:
    """
    直接拒絕的節點 (僅在 out_of_scope 時觸發)
    """
    print("[Node: refuse] 用戶詢問超出範圍，直接拒絕...")
    refusal_response = "抱歉，我目前的服務範圍僅限於『專案進度查詢』。您詢問的主題超出服務範疇，無法為您提供解答。"
    return {"response": refusal_response}

## 4. 定義路由函式 (Routing Function)

此函式將用於 `conditional_edges`，依據狀態中的 `intent` 值來決定流程走向。

In [ ]:
def route_by_intent(state: AgentState) -> Literal["retrieve_data", "refuse"]:
    """
    條件式路由：依據 intent 分流
    """
    if state["intent"] == "in_scope":
        return "retrieve_data"
    return "refuse"

## 5. 建立並編譯 Graph 工作流

利用 `StateGraph` 將所有的節點與邊（含條件式路由邊）串接。

In [ ]:
# 初始化 Workflow 並帶入定義好的狀態型別
workflow = StateGraph(AgentState)

# 註冊節點
workflow.add_node("classify_intent", classify_intent)
workflow.add_node("retrieve_data", retrieve_data)
workflow.add_node("generate_reply", generate_reply)
workflow.add_node("refuse", refuse)

# 設定起始邊
workflow.add_edge(START, "classify_intent")

# 設定條件式路由邊
workflow.add_conditional_edges(
    "classify_intent",
    route_by_intent,
    {
        "retrieve_data": "retrieve_data",
        "refuse": "refuse"
    }
)

# 設定一般邊
workflow.add_edge("retrieve_data", "generate_reply")
workflow.add_edge("generate_reply", END)
workflow.add_edge("refuse", END)

# 編譯工作流
app = workflow.compile()

## 6. 測試執行 (Testing)

我們測試兩種不同的輸入，觀察 Graph 的流向與最終輸出結果。

### 測試案例 A：符合服務範圍 (In Scope)

In [ ]:
print("=== 執行測試案例 A ===")
input_state_a: AgentState = {
    "user_input": "請幫我查一下專案 Alpha 目前的進度狀況。",
    "intent": "out_of_scope",  # 初始預設值，會被節點更新
    "retrieved_data": "",
    "response": ""
 }

result_a = app.invoke(input_state_a)
print("\n[最終回覆]:")
print(result_a["response"])

### 測試案例 B：超出服務範圍 (Out of Scope)

In [ ]:
print("=== 執行測試案例 B ===")
input_state_b: AgentState = {
    "user_input": "請問今天台北的天氣如何？會下雨嗎？",
    "intent": "out_of_scope",
    "retrieved_data": "",
    "response": ""
 }

result_b = app.invoke(input_state_b)
print("\n[最終回覆]:")
print(result_b["response"])